In [1]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
import os

from rdkit import Chem
from rdkit.Chem import rdmolops, AllChem, Draw
from openbabel import openbabel
from tqdm import tqdm

dataset_dir = '/Users/joeyshin/Desktop/THG/BioEdaDatabase/dataset'
df = pd.read_csv("halogen_EDA.csv", index_col=0)

In [5]:
alkyl = '[CX4:1]([F,Cl,Br,I:2])'
ring = '[c:1]-[F,Cl,Br,I:2]'

patts = [alkyl, ring]
patts = list(map(Chem.MolFromSmarts, patts))

def find_groups(mol, idx):    
    for index, patt in enumerate(patts):
        for sub in mol.GetSubstructMatches(patt):
            if sub[1] == (idx - 1):
                if index == 0:
                    C_idx = sub[0]
                    amount = len([sub for sub in mol.GetSubstructMatches(patt) if sub[0] == C_idx])
                    # if amount >= 1:
                    #     return 'alkyl'
                    if amount == 1:
                        return 'alkyl_1'
                    elif amount == 2:
                        return 'alkyl_2'
                    elif amount == 3:
                        return 'alkyl_3'
                elif index == 1:
                    return 'ring'
    return None

groups = []
groupID = []
errs = []

for index, row in tqdm(df.iterrows()):
    try:
        mol = Chem.SDMolSupplier(os.path.join(dataset_dir, f"{row['subdir']}{row['PDBID']}/{row['full_PDBID']}/{row['full_PDBID']}_ligand_refined.sdf"), removeHs=False)[0]
        groups.append(find_groups(mol, row['don_idx']))
        groupID.append([row['full_PDBID'], row['subdir'], find_groups(mol, row['don_idx'])])
    except Exception as e:
        errs.append((index, e))
        groups.append('')

df['group'] = groups

0it [00:00, ?it/s][00:42:07] Explicit valence for atom # 4 P, 6, is greater than permitted
[00:42:07] ERROR: Could not sanitize molecule ending on line 94
[00:42:07] ERROR: Explicit valence for atom # 4 P, 6, is greater than permitted
1318it [00:00, 2067.23it/s]


In [7]:
df_E = df.query('TOTAL < 0')

F_alkyl1 = df.query('donortype == "F" & TOTAL < 0 & group == "alkyl_1"')
F_alkyl2 = df.query('donortype == "F" & TOTAL < 0 & group == "alkyl_2"')
F_alkyl3 = df.query('donortype == "F" & TOTAL < 0 & group == "alkyl_3"')
F_ring = df.query('donortype == "F" & TOTAL < 0 & group == "ring"')
print('F: ')
print(len(F_alkyl1))
print(len(F_alkyl2))
print(len(F_alkyl3))
print(len(F_ring))

Cl_alkyl1 = df.query('donortype == "Cl" & TOTAL < 0 & group == "alkyl_1"')
Cl_alkyl2 = df.query('donortype == "Cl" & TOTAL < 0 & group == "alkyl_2"')
Cl_alkyl3 = df.query('donortype == "Cl" & TOTAL < 0 & group == "alkyl_3"')
Cl_ring = df.query('donortype == "Cl" & TOTAL < 0 & group == "ring"')
print('Cl: ')
print(len(Cl_alkyl1))
print(len(Cl_alkyl2))
print(len(Cl_alkyl3))
print(len(Cl_ring))

Br_alkyl1 = df.query('donortype == "Br" & TOTAL < 0 & group == "alkyl_1"')
Br_alkyl2 = df.query('donortype == "Br" & TOTAL < 0 & group == "alkyl_2"')
Br_alkyl3 = df.query('donortype == "Br" & TOTAL < 0 & group == "alkyl_3"')
Br_ring = df.query('donortype == "Br" & TOTAL < 0 & group == "ring"')
print('Br: ')
print(len(Br_alkyl1))
print(len(Br_alkyl2))
print(len(Br_alkyl3))
print(len(Br_ring))

I_alkyl1 = df.query('donortype == "I" & TOTAL < 0 & group == "alkyl_1"')
I_alkyl2 = df.query('donortype == "I" & TOTAL < 0 & group == "alkyl_2"')
I_alkyl3 = df.query('donortype == "I" & TOTAL < 0 & group == "alkyl_3"')
I_ring = df.query('donortype == "I" & TOTAL < 0 & group == "ring"')
print('I: ')
print(len(I_alkyl1))
print(len(I_alkyl2))
print(len(I_alkyl3))
print(len(I_ring))

F: 
7
15
186
168
Cl: 
1
1
0
498
Br: 
1
1
0
107
I: 
0
0
0
77


# Ring Halogen Distance